# Introduction to Natural Language Processing: Assignment 2

In this exercise we'll practice features extraction using Tf-Idf and SpaCy as well as multiclass text classification using the word embedding technique.

- You can use built-in Python packages, spaCy, scikit-learn, Numpy and Pandas.
- Please comment your code
- Submissions are due Tuesdays at 23:59 AoE **only** on eCampus: **Assignmnets >> Student Submissions >> Assignment 2**

- Name the file aproppriately: "Assignment_2_\<Your_Name\>.ipynb" and submit only the Jupyter Notebook file.
- If you are working in a group of two, please have the names of both of the members in the file name.
- Please use relative path, your code should work on my computer if the Jupyter Notebook and the file are both in the same directory.

Example: file_name = bbc-news.csv, **DON'T use:** /Users/ComputerName/Username/Documents/.../bbc-news.csv

### Task 1 (2 points)

Write a function `extract_proper_nouns(my_file_name)` that takes a file name (my_file_name.txt) as input and returns a list containing all proper nouns with more than one token.

**Example:**

text = "Hong Kong and Japan are two countries in Asia and New York is the largest city in the world"

return = `["New York", "Hong Kong"]` **(Note: it should not return "Japan")**

In [ ]:
def extract_proper_nouns(my_file_name):
    several_token_propn = []

    text = None

    with open(my_file_name, 'r') as file:
        text = file.read()

    words = text.split()

    saved= []

    for word in words:
        word = word.strip('.,!?;:"\'()[]{}1234567890')
        if word and word[0].isupper():       
            saved.append(word)
        else:
            if len(saved) > 1: # more than one token found
                several_token_propn.append(" ".join(saved))
                saved = []

    # flush
    if len(saved) > 1:
        several_token_propn.append(' '.join(saved))

    return(several_token_propn)

### Task 2 (3 points)

Write a function `common_lemma(my_file_name)` that takes a file name (my_file_name.txt) as input and returns a Python dictionary with lemmas as `key` and the `value` that should contain a list with both verbs and nouns sharing the same lemma.

**Examples:**

1.
text = "When users google for a word or any query, their system internally runs a pipeline in order to process what the person is querying."

return = `{"query": ["query", "querying"]}`

2.
text = I really loved the movie and show, the movie was showing reality but it showed sometimes nonesense!

return = `{"show": ["show", "showing", "showed"]}` **(Note: it should not return "movie" because both "movie"s are NOUN)**

In [ ]:
import spacy

nlp = spacy.load("en_core_web_sm") # load a spacy model

def common_lemma(my_file_name):
    with open(my_file_name, "r", encoding="utf-8") as f:
        text = f.read()

    lemma_info = {}

    for token in nlp(text):
        if token.pos_ not in ("NOUN", "VERB"):
            continue

        lemma = token.lemma_.lower()
        if lemma not in lemma_info:
            lemma_info[lemma] = {"pos": set(), "forms": []}

        lemma_info[lemma]["pos"].add(token.pos_)

        form = token.text.lower()
        if form not in lemma_info[lemma]["forms"]:
            lemma_info[lemma]["forms"].append(form)

    return {
        lemma: info["forms"]
        for lemma, info in lemma_info.items()
        if {"NOUN", "VERB"}.issubset(info["pos"])
    }

### Task 3 (1 point)

Load the data `bbc-text.csv`; This datata consists of 2225 documents from the BBC news website corresponding to stories from 2004-2005.

In [ ]:
# Here comes your code

import pandas as pd

df = pd.read_csv("bbc-news.csv")

### Task 4 (1 point)

Show how many articles we have for each topical area (class label) in the dataset using a plot.

In [ ]:
# Here comes your code

import matplotlib.pylab as plt

# Count articles in each class label
counts = df["category"].value_counts()

# Plot it
ax = counts.plot(kind="bar", figsize=(8, 5), color="steelblue")
ax.set_title("Articles per Topical Area")
ax.set_xlabel("topic")
ax.set_ylabel("Number of articles per topic")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

### Task 5 (2 point)

Preprocessing: Define two following functions and apply them to the dataset:
1. Remove punctuation
2. Remove any numbers

In [ ]:
import string


def remove_punctuation(corpus):
    cleaned_corpus = corpus.str.replace(f"[{string.punctuation}]", "", regex=True)
    return cleaned_corpus

def remove_numbers(corpus):
    cleaned_corpus = corpus.str.replace(r"\d+", "", regex=True)
    return cleaned_corpus

### Task 6.1 (1 points)

Split the data into training and test set (70% and 30%) using scikit-learn, shuffle it, and set the `seed=101 (random_state)`.

**NOTE:** If working with this dataset is not computationaly possible for you, you can work only with a subset of the dataset (i.e., the first 1000 rows) and use only the first 150 tokens for each article. You should point this out in your code.

In [ ]:
# Here comes your code

from sklearn.model_selection import train_test_split

X = df["text"]  # article text
y = df["category"]  # class label

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=True, random_state=101)

### Task 6.2 (3 points)

### **Training models on TF-IDF vectors:**

a) Convert each article in your data splits to a vector representation using the tf-idf-vectorizer.

b) Using the vectors from the previous step, train the `MLPClassifier` and another model of your choice from the scikit-learn library.

c) Test both of your models on the test set from Task 6.1.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

# a) Convert articles to TF-IDF vectors
tfidf_vectorizer = TfidfVectorizer()

X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# b) Train the models
mlp_tfidf = MLPClassifier(
    hidden_layer_sizes=(100,),
    max_iter=300,
    random_state=101
)
mlp_tfidf.fit(X_train_tfidf, y_train)

logreg_tfidf = LogisticRegression(max_iter=1000, random_state=101)
logreg_tfidf.fit(X_train_tfidf, y_train)

# c) Predict labels for the test set
y_pred_MLP_tfidf = mlp_tfidf.predict(X_test_tfidf)
y_pred_LogisticRegression_tfidf = logreg_tfidf.predict(X_test_tfidf)

### Task 6.3 (3 points)

### **Training models on SpaCy model vector representation:**

a) Convert each article in your data splits to a vector representation using the pre-trained spaCy model. (**Hint:** It should be stored as an array)

b) Using the vectors from the previous step, train the `MLPClassifier` and another model of your choice from the scikit-learn library.

c) Test both of your models on the test set from Task 6.1.

In [ ]:
# Here comes your code
import numpy as np
import spacy
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

nlp = spacy.load("en_core_web_lg")


def spacy_vectors(texts):
    return np.vstack([doc.vector for doc in nlp.pipe(texts, batch_size=64)])


# a) Convert to array of spacy vectors
X_train_spacy = np.vstack([doc.vector for doc in nlp.pipe(X_train, batch_size=64)])
X_test_spacy = np.vstack([doc.vector for doc in nlp.pipe(X_test, batch_size=64)])

# b) Train models
mlp_spacy = MLPClassifier(hidden_layer_sizes=(100,), max_iter=1000, random_state=101)
mlp_spacy.fit(X_train_spacy, y_train)

logreg_spacy = LogisticRegression(max_iter=2000, random_state=101)
logreg_spacy.fit(X_train_spacy, y_train)

# c) Predict labels for test set
y_pred_MLP_spacy = mlp_spacy.predict(X_test_spacy)
y_pred_LogisticRegression_spacy = logreg_spacy.predict(X_test_spacy)


### Task 7 (4 points)

Using the predictions from the four classifiers, evaluate the models and report accuracy, recall, precision, f1 scores and confusion matrix for each of them. (**Hint:** You should build a confusion matrix for multi-class classification)

In [ ]:
# Here comes your code
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

predictions = {
    "MLP (TF-IDF)": y_pred_MLP_tfidf,
    "Logistic Regression (TF-IDF)": y_pred_LogisticRegression_tfidf,
    "MLP (spaCy)": y_pred_MLP_spacy,
    "Logistic Regression (spaCy)": y_pred_LogisticRegression_spacy,
}

labels = sorted(y_test.unique())

for model_name, y_pred in predictions.items():
    print(f"\n{model_name}")
    print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f}")
    print(f"Precision: {precision_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"Recall:    {recall_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")
    print(f"F1 score:  {f1_score(y_test, y_pred, average='weighted', zero_division=0):.4f}")

    print("\nConfusion matrix (rows = true labels, columns = predicted labels):")
    print(confusion_matrix(y_test, y_pred, labels=labels))

    print("\nPer-class report:")
    print(classification_report(y_test, y_pred, labels=labels, zero_division=0))